In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("YELP_API_KEY")

print (f"This is the key: {api_key}")   


This is the key: _TZZJbs55ENUDK2lRDU0lwmVf1DodAq4ja16ahgD9C_dd1WA8o1bEiCBetWi9cmFS9tdWiH5GKzAUKSh97-RjA62cALkEmAT4VslMrSdO1CzE8FFYVQdMZxw2-JLaHYx


In [5]:
# ✅ Yelp API Pull (Jupyter Version)

from pathlib import Path
from datetime import datetime
import os, json, requests
from dotenv import load_dotenv

# For Jupyter, use cwd walk-up instead
while not Path("README.md").exists() and Path.cwd() != Path("/"):
    os.chdir("..")  # Keep going up until we hit project root (assumes README is there)

# Now safely set paths
raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

# Make sure you’ve exported your key in terminal before launching Jupyter:
# export YELP_API_KEY='your-key'
load_dotenv()
YELP_API_KEY = os.getenv("YELP_API_KEY")

headers = {
    "Authorization": f"Bearer {YELP_API_KEY}"
}

params = {
    "term": "asian food",
    "location": "Irvine, CA",
    "limit": 20,
    "sort_by": "rating"
}

response = requests.get("https://api.yelp.com/v3/businesses/search", headers=headers, params=params)

if response.status_code == 200:
    businesses = response.json().get("businesses", [])
    


    # Save as newline-delimited JSON
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    # filename = f"yelp_sample_{timestamp}.json" old line where it would save it in the current directory
    filename = raw_dir / f"yelp_sample_{timestamp}.json"


    with open(filename, "w") as f:
        for business in businesses:
            json.dump(business, f)
            f.write("\n")
    
    print(f"✅ JSON file saved as {filename}")
else:
    print("❌ Yelp API call failed:", response.status_code, response.text)


✅ JSON file saved as data/raw/yelp_sample_20250619_202744.json


In [4]:
from pathlib import Path
import os
import glob
from google.cloud import storage
from dotenv import load_dotenv
load_dotenv()


os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Dynamically move to project root (assuming README.md is there as a marker)
while not Path("README.md").exists() and Path.cwd() != Path("/"):
    os.chdir("..")

# Now find latest .json in project-root-relative data/raw/
json_files = glob.glob("data/raw/*.json")

if not json_files:
    raise FileNotFoundError("No JSON files found in data/raw/. Make sure you've run the API pull.")

latest_file = max(json_files, key=os.path.getctime)

# Upload to GCS
bucket_name = "daily-restaurant-insights-bucket"  # ← change this
destination_blob_path = f"raw/{os.path.basename(latest_file)}"

client = storage.Client()
bucket = client.bucket(bucket_name)
blob = bucket.blob(destination_blob_path)
blob.upload_from_filename(latest_file)

print(f"✅ Uploaded {latest_file} to gs://{bucket_name}/{destination_blob_path}")


✅ Uploaded data/raw/yelp_sample_20250619_202744.json to gs://daily-restaurant-insights-bucket/raw/yelp_sample_20250619_202744.json


In [ ]:
### This Script loads the latest yelp api pull into storage bucket and then loads the data into bronze layer table.

from google.cloud import bigquery
from google.cloud import storage
from google.oauth2 import service_account
from datetime import datetime

import os
import glob



# ✅ 1. Define constants
bucket_name = "daily-restaurant-insights-bucket"  # Change if needed
project_id = "daily-restaurant-insights"  # 🔁 Replace with your actual GCP project ID
dataset_id = "daily_restaurant_insights"
table_name = "bronze_yelp_raw"

# ✅ 2. Navigate from current notebook folder to root/data/raw
notebook_path = os.path.abspath("")  # current notebook directory
project_root = os.path.dirname(notebook_path)  # one level up
raw_folder = os.path.join(project_root, "data", "raw")

# Path to your JSON key
key_path = os.path.join(project_root, "keys", "daily-restaurant-insights-0659005580cc.json")

# Create credentials object
credentials = service_account.Credentials.from_service_account_file(key_path)


# ✅ 3. Find most recent file in raw data
json_files = glob.glob(os.path.join(raw_folder, "*.json"))
latest_file = max(json_files, key=os.path.getctime)

if not json_files:
    raise FileNotFoundError("⚠️ No JSON files found in data/raw/")

# ✅ 4. Upload to Google Cloud Storage
destination_blob_path = f"raw/{os.path.basename(latest_file)}"
storage_client = storage.Client(credentials=credentials)
bucket = storage_client.bucket(bucket_name)
blob = bucket.blob(destination_blob_path)

try:
    blob.upload_from_filename(latest_file)
    print(f"✅ Uploaded {latest_file} to gs://{bucket_name}/{destination_blob_path}")
except Exception as e:
    print(f"❌ Upload to GCS failed: {e}")
    raise


# ✅ 5. Load into BigQuery
table_id = f"{project_id}.{dataset_id}.{table_name}"
gcs_uri = f"gs://{bucket_name}/{destination_blob_path}"

bq_client = bigquery.Client(credentials=credentials, project=project_id)


job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
    autodetect=True,  # Auto-detect schema from the JSON
    write_disposition="WRITE_APPEND"  # Append mode
)

load_job = bq_client.load_table_from_uri(
    gcs_uri,
    table_id,
    job_config=job_config
)


try:
    load_job = bq_client.load_table_from_uri(
        gcs_uri,
        table_id,
        job_config=job_config
    )
    load_job.result()
    print(f"✅ Data loaded into BigQuery table: {table_id}")
except Exception as e:
    print(f"❌ BigQuery load failed: {e}")
    raise



✅ Uploaded /home/devuser/data/raw/yelp_sample_20250619_202744.json to gs://daily-restaurant-insights-bucket/raw/yelp_sample_20250619_202744.json
✅ Data loaded into BigQuery table: daily-restaurant-insights.daily_restaurant_insights.bronze_yelp_raw
